In [1]:
import balancepy as bp
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import balancepy.model_sim.P18 as P18
from pprint import pprint

In [2]:
# Define subject antrhopometry
weight = 53
height = 1.66

# create a model instance for the subject
model = P18.P18(weight, height)

# show default model parameters (not tuned to experimental data)
model

balancepyModel(ModelName=Peterka2018,
  ParameterSet( 
    mgh: value=8.05437768699842, bounds=(10, 20), fixed=True,
    J: value=0.9021304550652025, bounds=(0, 0), fixed=True,
    Kp: value=11.678847646147707, bounds=(8.457096571348341, 20.13594421749605), fixed=False,
    Kd: value=3.5439261822793044, bounds=(0.805437768699842, 8.05437768699842), fixed=False,
    W: value=0.45, bounds=(0.01, 1), fixed=False,
    dt: value=0.16, bounds=(0.1, 0.3), fixed=False,
    Kt: value=0.005, bounds=(0, 0.05), fixed=False
    ),
  Frequencies=200 frequencies from 0.01 to 2.0,
  Fit Reference=Not defined,
  FRF Smoothing=<function P18.<lambda> at 0x13f26c900>,
  Sampling Rate=90 Hz)

In [3]:
tf = model.transfer_function()
print("Transfer Function:")
print(tf)

Transfer Function:
TransferFunctionContinuous(
array([-1.73699947, 15.98829054, 71.5525348 ,  0.        ]),
array([ 1.        ,  8.26210091, 25.88414139, 50.77741718,  6.40345711]),
dt: None
)


In [4]:
model.plot()

No experimental data available for plotting


In [5]:
# Run time domain simulations

# Define the stimulus
samplingrate = 90  # Hz
stim = bp.get_mseq(type="prts80", samplingrate=samplingrate)
time = np.arange(0, len(stim) / samplingrate, 1 / samplingrate)

# Set stimulus and samplingrate in model
model.samplingrate = samplingrate
model.stimulus = stim

# Run time domain simulation
TDsim = model.simulate_TD()

model.plot()


No experimental data available for plotting


In [6]:
# Add reference experimental data to model and perform fit

# Load experimental data
com, stim_vis, stim_prop, time = bp.getdata_lifespan('data/d1_vis1_surf1.csv', height, resample=True, cut_to_cycles=True)

model.add_experimental_data(stim_prop, com, 90)

model.fit()
figure = model.plot()

pprint(model.params.to_value_dict())
figure.show()

{'J': 0.9021304550652025,
 'Kd': np.float64(3.8006853671975303),
 'Kp': np.float64(11.118943645489088),
 'Kt': np.float64(0.011962347966016681),
 'W': np.float64(0.5906184487724994),
 'dt': np.float64(0.14933198923681906),
 'mgh': 8.05437768699842}


In [ ]:
# complete steps for 10s long visual stimulus
# Define subject antrhopometry
weight = 53
height = 1.66

config = {
    "ModelName": "Peterka2018",
    "frfSmoothing": lambda x, f: bp.logspace_manual_10s(x,f)
}

# create a model instance for the subject
model_vis = P18.P18(weight, height, config=config)
# Load experimental data
com, stim_vis, stim_prop, time = bp.getdata_lifespan(
    'data/d1_vis1.csv', 
    height, 
    resample=True, 
    cut_to_cycles=True,
    end_time=220,
    cycle_start_samples=20*90,
    cycle_length_samples=10*90
)

model_vis.params['W'] = 0.1
model_vis.add_experimental_data(stim_vis, com, 90)

model_vis.fit()

print(model_vis)

figure = model_vis.plot()
figure.show()

balancepyModel(ModelName=Peterka2018,
  ParameterSet( 
    mgh: value=8.05437768699842, bounds=(10, 20), fixed=True,
    J: value=0.9021304550652025, bounds=(0, 0), fixed=True,
    Kp: value=11.678847646147707, bounds=(8.457096571348341, 20.13594421749605), fixed=False,
    Kd: value=3.5439261822793044, bounds=(0.805437768699842, 8.05437768699842), fixed=False,
    W: value=0.1, bounds=(0.01, 1), fixed=False,
    dt: value=0.16, bounds=(0.1, 0.3), fixed=False,
    Kt: value=0.005, bounds=(0, 0.05), fixed=False
    ),
  Frequencies=9 frequencies from 0.1 to 1.7000000000000002,
  Fit Reference=Defined,
  FRF Smoothing=<function <lambda> at 0x146db7c40>,
  Sampling Rate=90 Hz)
balancepyModel(ModelName=Peterka2018,
  ParameterSet( 
    mgh: value=8.05437768699842, bounds=(10, 20), fixed=True,
    J: value=0.9021304550652025, bounds=(0, 0), fixed=True,
    Kp: value=8.457096571348341, bounds=(8.457096571348341, 20.13594421749605), fixed=False,
    Kd: value=4.071559308287843, bounds=(0.8054